In [1]:
!pip install spacy scikit-learn seaborn matplotlib plotly nbformat umap-learn

  Using cached spacy-3.8.16-cp312-cp312-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (28 kB)
  Using cached scikit_learn-1.9.1-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (9.3 kB)
  Using cached seaborn-0.13.2-py3-none-any.whl.metadata (5.4 kB)
  Using cached matplotlib-3.11.2-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (80 kB)
  Using cached plotly-7.1.0-py3-none-any.whl.metadata (8.8 kB)
  Using cached nbformat-5.11.1-py3-none-any.whl.metadata (3.7 kB)
  Using cached umap_learn-0.5.12-py3-none-any.whl.metadata (24 kB)
  Using cached spacy_legacy-3.0.12-py2.py3-none-any.whl.metadata (2.8 kB)
  Using cached spacy_loggers-1.0.5-py3-none-any.whl.metadata (23 kB)
  Using cached murmurhash-1.0.15-cp312-cp312-manylinux1_x86_64.manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_5_x86_64.whl.metadata (2.3 kB)
  Using cached cymem-2.0.13-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (9.7 kB)
  Using cached p

In [2]:
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 3.8 MB/s eta 0:00:0000:0100:01
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


# 2. Data preparation

## Text lemmatization with spacy

In [3]:
import spacy
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from tqdm import tqdm

import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px

nlp = spacy.load('en_core_web_sm')

In [6]:
dataset = pd.read_csv('.data/dataset.csv')

In [7]:
all_song_lyrics = []
for song in tqdm(dataset['Lyrics'].tolist()):
    clean_lyrics = []
    for line in song.split('\n'):
        temp = []
        d = nlp(line)
        for word in d:
            if not word.is_punct:
                temp.append(word.lemma_)
        clean_lyrics.append(' '.join(temp))
    all_song_lyrics.append(clean_lyrics)

100%|██████████| 373/373 [01:28<00:00,  4.23it/s]


In [8]:
dataset['Lyrics_lemmatized'] = all_song_lyrics
dataset.Lyrics_lemmatized.iloc[0][:10]

['I be in my new house place daydream',
 'patch up the crack along the wall',
 'I pass it and lose track of what I be say',
 "'cause that be where I be when I get the call",
 "when the first stone 's throw there be screamin",
 'in the street there be a ragin riot',
 'when it be burn the bitch they be shriek',
 'when the truth come out it be quiet',
 'so they kill Cassandra first',
 "'cause she fear the bad"]

## N-gram vectorizer on song level: identifying identical lines and leaving only one

In [9]:
corpus = dataset.Lyrics_lemmatized.iloc[0]
ngram_vectorizer = CountVectorizer(analyzer='word', ngram_range=(1, 1))
X2 = ngram_vectorizer.fit_transform(corpus)
X2 = X2.toarray()
similarity_matrix_song = cosine_similarity(X2)
similarity_matrix_song.shape

(51, 51)

In [10]:
corpus_hover = []
for x in range(len(corpus)):
    corpus_hover.append(
        list(map(lambda l: f'<br>X={l}</br><br>Y={corpus[x]}</br>',
         corpus))
    )

In [12]:
fig = px.imshow(
    similarity_matrix_song,
    labels=dict(x="X", y="Y", color="Intensity"),
    x=[i for i in range(similarity_matrix_song.shape[0])],
    y=[i for i in range(similarity_matrix_song.shape[0])],
    #text_auto=True,                # Automatically displays values inside the boxes
    color_continuous_scale="Viridis" # Choose color mapping style
)

fig.update_traces(
    customdata=corpus_hover,
    hovertemplate="<br>X=%{x}; Y=%{y}</br>%{customdata}<br>Similarity:%{z}</br><extra></extra>"
)
fig.show()

html_str = fig.to_html(full_html=False, include_plotlyjs='cdn')

with open("figures/cassandra_before.html", "w") as f:
    f.write(html_str)

In [78]:
def delete_reduntant_lines(loc):
    ngram_vectorizer = CountVectorizer(analyzer='word', ngram_range=(1, 1))
    X2 = ngram_vectorizer.fit_transform(loc)
    X2 = X2.toarray()
    similarity_matrix_song = cosine_similarity(X2)

    #lines_stay = []
    lines_delete = np.array([])
    appended_lines = []
    
    for line_i in range(X2.shape[0]):
        similarity_line = similarity_matrix_song[line_i]
        similar_lines = np.nonzero(similarity_line > 0.8)[0]
        #print(line_i, similar_lines)
        if len(similar_lines) > 1:
            if similar_lines[0] not in appended_lines:
                #lines_stay.append(similar_lines[0])
                lines_delete = np.append(lines_delete, similar_lines[1:])
                appended_lines += list(similar_lines)
    array = np.arange(0, loc.__len__())
    lines_to_save = array[np.isin(array, lines_delete,
              invert=True)]
    # print('\n'.join([loc[i] for i in array[np.isin(array, lines_delete,
    #           invert=False)]]))
    return [loc[i] for i in lines_to_save]
song_no_redundant = delete_reduntant_lines(dataset.Lyrics_lemmatized.iloc[0])

In [79]:
X_new = ngram_vectorizer.fit_transform(song_no_redundant)
similarity_matrix_song_new = cosine_similarity(X_new)

corpus_hover_new = []
for x in range(len(song_no_redundant)):
    corpus_hover_new.append(
        list(map(lambda l: f'<br>X={l}</br><br>Y={song_no_redundant[x]}</br>',
         song_no_redundant))
    )

fig = px.imshow(
    similarity_matrix_song_new,
    labels=dict(x="X", y="Y", color="Intensity"),
    x=[i for i in range(similarity_matrix_song_new.shape[0])],
    y=[i for i in range(similarity_matrix_song_new.shape[0])],
    #text_auto=True,                # Automatically displays values inside the boxes
    color_continuous_scale="Viridis" # Choose color mapping style
)

fig.update_traces(
    customdata=corpus_hover_new,
    hovertemplate="<br>X=%{x}; Y=%{y}</br>%{customdata}<br>Similarity:%{z}</br><extra></extra>"
)
fig.show()

In [72]:
dataset['Lyrics_no_redundant'] = dataset.Lyrics.apply(lambda x: x.split('\n')).apply(delete_reduntant_lines).apply(lambda x: '\n'.join(x))
dataset['Lyrics_lemmatized_no_redundant'] = dataset.Lyrics_lemmatized.apply(delete_reduntant_lines).apply(lambda x: '\n'.join(x))

In [14]:
dataset.to_csv('.data/dataset_no_reduntant.csv', index=False)

## Finding similar lines across the full corpus

In [80]:
full_corpus = []
song_sames = []
for _ in range(dataset.shape[0]):
    full_corpus += dataset.Lyrics_lemmatized_no_redundant.iloc[_].split('\n')
    song_sames += [dataset['Song Name'].iloc[_]] * len(dataset.Lyrics_lemmatized_no_redundant.iloc[_].split('\n'))
full_corpus.__len__(), full_corpus[:10]

(11442,
 ['I be in my new house place daydream',
  'patch up the crack along the wall',
  'I pass it and lose track of what I be say',
  "'cause that be where I be when I get the call",
  "when the first stone 's throw there be screamin",
  'in the street there be a ragin riot',
  'when it be burn the bitch they be shriek',
  'when the truth come out it be quiet',
  'so they kill Cassandra first',
  "'cause she fear the bad"])

In [74]:
ngram_vectorizer = CountVectorizer(analyzer='word', ngram_range=(1, 1))
X_full = ngram_vectorizer.fit_transform(full_corpus)
X_full = X_full.toarray()
similarity_matrix_full = cosine_similarity(X_full)

In [17]:
similar_lines_corpus = []
with open('similar_lines.txt' , 'w') as f:
    for i in range(len(full_corpus)):
        similarity_line = similarity_matrix_full[i]
        if len(np.nonzero(similarity_line > 0.98)[0]) > 1:
            print('Song: ', song_sames[i], '\t', 'Line: ', full_corpus[i], file=f)
            similar_lines_corpus.append(full_corpus[i])
            for _ in np.nonzero(similarity_line > 0.9)[0]:
                if _ != i:
                    print('Song: ', song_sames[_], '\t', 'Line: ', full_corpus[_], 'Similarity: ', round(similarity_line[_], 3), file=f)
                    similar_lines_corpus.append(full_corpus[_])
            print('\n', file=f)

In [137]:
def delete_reduntant_lines(loc):
    ngram_vectorizer = CountVectorizer(analyzer='word', ngram_range=(1, 1))
    X2 = ngram_vectorizer.fit_transform(loc)
    X2 = X2.toarray()
    similarity_matrix_song = cosine_similarity(X2)

    lines_stay = []
    lines_delete = np.array([])
    appended_lines = []
    
    for line_i in range(X2.shape[0]):
        similarity_line = similarity_matrix_song[line_i]
        similar_lines = np.nonzero(similarity_line > 0.8)[0]
        if len(similar_lines) == 1:
            lines_stay.append(similar_lines[0])
        if len(similar_lines) > 1:
            lines_stay.append(similar_lines[0])
            if not np.intersect1d(similar_lines[1:], lines_delete).shape[0] == similar_lines[1:].shape[0]:
                lines_delete = np.append(lines_delete, similar_lines[1:])
        unique_words = set(loc[line_i].split(' '))
        len_unique_words = list(map(lambda x: len(x), 
                                    unique_words))
        if len(unique_words) < 2:
            lines_delete = np.append(lines_delete, line_i)
        if np.array(len_unique_words).mean() <= 2.5:
            lines_delete = np.append(lines_delete, line_i)
        if np.array(len_unique_words).mean() <= 4:
            if 'yeah' in unique_words or 'ooh' in unique_words:
                lines_delete = np.append(lines_delete, line_i)
        if loc[line_i] in ['Taylor Swift', 'Ed Sheeran', 'Justin Vernon']:
            lines_delete = np.append(lines_delete, line_i)
    array = np.arange(0, loc.__len__())
    lines_to_save = array[np.isin(array, lines_delete,
              invert=True)]
    return [loc[i] for i in lines_to_save]

In [141]:
dataset['Lyrics_no_redundant'] = dataset.Lyrics.apply(lambda x: x.split('\n')).apply(delete_reduntant_lines).apply(lambda x: '\n'.join(x))
dataset['Lyrics_lemmatized_no_redundant'] = dataset.Lyrics_lemmatized.apply(delete_reduntant_lines).apply(lambda x: '\n'.join(x))
dataset.to_csv('.data/dataset_no_reduntant.csv')

In [140]:
full_corpus = []
song_sames = []
for _ in range(dataset.shape[0]):
    full_corpus += dataset.Lyrics_lemmatized_no_redundant.iloc[_].split('\n')
    song_sames += [dataset['Song Name'].iloc[_]] * len(dataset.Lyrics_lemmatized_no_redundant.iloc[_].split('\n'))

ngram_vectorizer = CountVectorizer(analyzer='word', ngram_range=(1, 1))
X_full = ngram_vectorizer.fit_transform(full_corpus)
X_full = X_full.toarray()
similarity_matrix_full = cosine_similarity(X_full)

similar_lines_corpus = []
with open('similar_lines_after_filter.txt' , 'w') as f:
    for i in range(len(full_corpus)):
        similarity_line = similarity_matrix_full[i]
        if len(np.nonzero(similarity_line > 0.98)[0]) > 1:
            print('Song: ', song_sames[i], '\t', 'Line: ', full_corpus[i], file=f)
            similar_lines_corpus.append(full_corpus[i])
            for _ in np.nonzero(similarity_line > 0.9)[0]:
                if _ != i:
                    print('Song: ', song_sames[_], '\t', 'Line: ', full_corpus[_], 'Similarity: ', round(similarity_line[_], 3), file=f)
                    similar_lines_corpus.append(full_corpus[_])
            print('\n', file=f)